In [ ]:
import subprocess
import os

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300
from matplotlib.pyplot import figure, show, subplots
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.lines import Line2D

from scipy.interpolate import RegularGridInterpolator
import numpy as np


#import yaml
from ruamel.yaml import YAML
yaml = YAML()
yaml.preserve_quotes = True  # Keep quotes if they exist
yaml.indent(mapping=2, sequence=4, offset=2)  # Preserve indentation
yaml.representer.add_representer(
    float, lambda dumper, data: dumper.represent_scalar("tag:yaml.org,2002:float", str(data))
)
with open("../config/masterconfig.yaml", "r") as file:
    config = yaml.load(file) or {}
integrationtimes = config['integrationtimes']
observingdistances = config['observingdistances']
dataserver_path = config['dataserver_path']

In [ ]:
# this starts the retrievals

def ARCiS_initialize(atmosphere_key):
      splits = atmosphere_key.split("_")
      distance = int(splits[-2])
      
      config_ARCiS_loc = "config/config_ARCiS.in"
      save_location = f"/dataserver/users/formingworlds/borgmann/ARCiS_retrievals/{atmosphere_key}"
      atmosphere_file_loc = f"/Users/users/borgmann/Documents/masterproject/LIFEredoxsurvey/output/ARCiS/forwardmodel_ARCiS/textfiles/ARCiS_fm_input_{atmosphere_key}.txt"
      atmosphere_key_us = atmosphere_key.replace(".","_")
      command = f"ARCiS {config_ARCiS_loc} -o {save_location} -s obs1:file={atmosphere_file_loc} -s distance={distance}"
      prompt = (
        "ssh -n -T norma2;"
        "cd .. ;"
        f"tmux new-session -d -s '{atmosphere_key}'; "
        f"tmux send-keys -t {atmosphere_key_us}:0 'module unload ARCiS' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 'module load ARCiS' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 'module unload cfitsio' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 'module load cfitsio' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 'export OMP_NUM_THREADS=8' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 '{command}' C-m"
        )
      subprocess.run(prompt, shell=True, check=True) 

In [ ]:
# The atmosphere keys are defined as fo2_CH_observingdistance_integrationtime

init_list1 = ["-4.0_0.1_15_10","-4.0_1.0_15_10","-4.0_5.0_15_10"]
init_list2 = ["-1.33_0.1_15_10","-1.33_1.0_15_10","-1.33_5.0_15_10"]
init_list3 = ["1.33_0.1_15_10","1.33_1.0_15_10","1.33_5.0_15_10"]
init_list4 = ["4.0_0.1_15_10","4.0_1.0_15_10","4.0_5.0_15_10"]
init_list5 = ["-4.0_0.1_20_10","-4.0_1.0_20_10","-4.0_5.0_20_10"]
init_list6 = ["-1.33_0.1_20_10","-1.33_1.0_20_10","-1.33_5.0_20_10"]
init_list7 = ["1.33_0.1_20_10","1.33_1.0_20_10","1.33_5.0_20_10"]
init_list8 = ["4.0_0.1_20_10","4.0_1.0_20_10","4.0_5.0_20_10"]
for i in init_list6:
    ARCiS_initialize(i)

In [ ]:
# This checks automatically for any finished runs in the output folder and then starts the postprocessing

def ARCiS_postprocess(atmosphere_key):
    folder_location = f"/dataserver/users/formingworlds/borgmann/ARCiS_retrievals/{atmosphere_key}/"

    # check if post-processing has already been done
    if os.path.isfile(folder_location + "retrieval") == True:
        print("Post-processing is already done!")
    else:
        atmosphere_key_us = atmosphere_key.replace(".","_")
        command = f"ARCiS {folder_location}bestfit.dat -o {folder_location} -s pew=.true. -s Pmax=1000"
        prompt = (
        "ssh -n -T norma2;"
        f"tmux new-session -d -s 'pp_{atmosphere_key}'; "
        f"tmux send-keys -t pp_{atmosphere_key_us}:0 'module unload ARCiS' C-m ;"
        f"tmux send-keys -t pp_{atmosphere_key_us}:0 'module load ARCiS' C-m ;"
        f"tmux send-keys -t pp_{atmosphere_key_us}:0 'module unload cfitsio' C-m ;"
        f"tmux send-keys -t pp_{atmosphere_key_us}:0 'module load cfitsio' C-m ;"
        f"tmux send-keys -t pp_{atmosphere_key_us}:0 'export OMP_NUM_THREADS=8' C-m ;"
        f"tmux send-keys -t pp_{atmosphere_key_us}:0 '{command}' C-m"
        )
        subprocess.run(prompt, shell=True, check=True) 

In [ ]:
init_list1 = ["-4.0_0.1_15_10","-4.0_1.0_15_10","-4.0_5.0_15_10"]
init_list2 = ["-1.33_0.1_15_10","-1.33_1.0_15_10","-1.33_5.0_15_10"]
init_list3 = ["1.33_0.1_15_10","1.33_1.0_15_10","1.33_5.0_15_10"]
init_list4 = ["4.0_0.1_15_10","4.0_1.0_15_10","4.0_5.0_15_10"]
init_list5 = ["-4.0_0.1_20_10","-4.0_1.0_20_10","-4.0_5.0_20_10"]
init_list6 = ["-1.33_0.1_20_10","-1.33_1.0_20_10","-1.33_5.0_20_10"]
init_list7 = ["1.33_0.1_20_10","1.33_1.0_20_10","1.33_5.0_20_10"]
init_list8 = ["4.0_0.1_20_10","4.0_1.0_20_10","4.0_5.0_20_10"]
for atmosphere in init_list6:
    ARCiS_postprocess(atmosphere)

In [ ]:
# you can run this to check which runs are done and which are still running

def tmux_checkstatus():
    command = """for pane in $(tmux list-panes -a -F '#{session_name}:#{window_index}.#{pane_index}'); do
    if tmux capture-pane -p -t "$pane" | grep -q "All done!"; then
        echo "$pane: DONE"    
    else
        echo "$pane: still running"
    fi
    done"""
    prompt = "ssh -n -T norma2;" + command
    subprocess.run(prompt, shell=True, check=True)

tmux_checkstatus()

In [ ]:
# This collects all the retrieval runs that are done and postprocessed
# they are only done and postprocessed when the file called 'retrieval' is produced

done = []
data_location = dataserver_path + "/ARCiS_retrievals/"
atmosphere_folders = os.listdir(data_location)
for i in atmosphere_folders:
    if os.path.isfile(data_location+i+"/retrieval"):
        done.append(i)

            
volatile_labels = np.genfromtxt(data_location+done[0]+"/retrieval",usecols=0,dtype='str')[1:11]
print(volatile_labels)

atmosphere_data = []
for i in range(len(done)):
    medians = np.genfromtxt(data_location+done[i]+"/retrieval",usecols=1)[1:11]
    medians = medians/sum(medians)
    atmosphere_data.append([[float(j) for j in done[i].split("_")],medians])

atmosphere_dict = {(x,y,z,t): arr for [x,y,z,t], arr in atmosphere_data}
print(atmosphere_dict[(-1.33,5.0,10.0,350.0)])


In [ ]:
volatile_labels = ['N2', 'CO', 'H2', 'CH4', 'H2S', 'SO2', 'H2O', 'CO2', 'NH3', 'OCS']
volatile_colours = ["gray","magenta","red","green","pink","orange","blue","cyan","purple","navy"]

def plot_retrievals(select_distance,select_observingtime):
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_xlim(-5, 5)
    ax.set_ylim(0.05,10)
    ax.set_yscale('log')

    for keys in atmosphere_dict:
        fo2 = keys[0]
        CH = keys[1]
        distance = keys[2]
        if distance != select_distance:
            continue
        observingtime = keys[3]
        if observingtime != select_observingtime:
            continue
        fractions = atmosphere_dict[keys]

        # Create an inset axis at data coordinate
        inset = inset_axes(ax, width=1.2, height=1.2, 
                        bbox_to_anchor=(float(fo2), float(CH)),
                        bbox_transform=ax.transData, 
                        loc='center')

        # Draw a pie chart in the inset axis
        inset.pie(fractions, labels=None,
            colors=volatile_colours)
        inset.set_aspect('equal')

    # add legend
    custom_lines = []
    for i in volatile_colours:
        custom_lines.append(Line2D([0], [0], color=i, lw=4))
    ax.legend(custom_lines,volatile_labels,bbox_to_anchor=(1.15, 1.0))
    ax.set_xlabel("Redox state")
    ax.set_ylabel("C/H ratio")

    plt.show()
    fig.savefig('../output/figures/retrievals_{}hrs_{}pc.png'.format(select_observingtime,select_distance),bbox_inches='tight')

In [ ]:
for select_observingtime in integrationtimes:
    for select_distance in observingdistances:
        plot_retrievals(select_distance,round(select_observingtime/60/60))

In [ ]:
# Extra plot that I did not use in the thesis
# Shows the ARCiS forward models with the ARCiS retrievals with 1 sigma uncertainty

def spectrum_cutoff(bandcenter,bandflux):
    """Cut spectrum to only include 4-18.5 microns"""
    for i in range(len(bandcenter)):
        if bandcenter[i] > 4.0:
            lower_index = i-1
            break
    for j in range(len(bandcenter)):
        if bandcenter[j] > 18.5:
            upper_index = j+1
            break
    return bandcenter[lower_index:upper_index], bandflux[lower_index:upper_index]


done = []
data_location = "/dataserver/users/formingworlds/borgmann/ARCiS_forwardmodels/"
atmosphere_folders = os.listdir(data_location)
for i in atmosphere_folders:
    if os.path.isfile(data_location+i+"/emis"):
            done.append(i)

def sort_key(name):
    parts = name.replace("fm_ARCiS", "").split("_")
    fo2 = float(parts[0])
    CH = float(parts[1])
    dist = float(parts[2])
    return (fo2, CH, dist)

done = sorted(done, key=sort_key)


data_path = "/dataserver/users/formingworlds/borgmann/ARCiS_forwardmodels"
files = [f for f in os.listdir(data_path)]

mpl.rcParams['figure.dpi'] = 800
rasterplot = figure(figsize=(8,6))

# setup custom order for rasterplot, since the subplots go left->right, up->down
# but we need to go down->up, left->right
custom_order = [9,5,1,10,6,2,11,7,3,12,8,4]

custom_count = 0
# do plots
for i in range(len(done)):
   if done[i].split("_")[-1] == '10':
        print(custom_count)
        frame = rasterplot.add_subplot(3,4,custom_order[custom_count])
        bandcenter = np.genfromtxt(data_location+done[i]+"/emis",usecols=0,skip_header=1)
        bandflux = np.genfromtxt(data_location+done[i]+"/emis",usecols=1,skip_header=1)
        bandcenter, bandflux = spectrum_cutoff(bandcenter,bandflux)
        frame.plot(bandcenter,bandflux,linewidth=0.5,label='known')
        frame.set_yscale('log')
        frame.set_xlim(4,18.5)
        frame.set_ylim(10e-10,2.4122010236909366e-06)
        if custom_count in [1,2]:
            plt.tick_params(
                axis='both',      # both x and y axes
                which='both',     # major and minor ticks
                labelsize=6,      # ↓ font size of tick labels
                length=0,         # ↓ length of tick marks
                width=0.4,         # ↓ thickness of tick marks
                pad=0.8
            )
            frame.minorticks_off()
        elif custom_count == 0:
            plt.tick_params(
                axis='both',      # both x and y axes
                which='both',     # major and minor ticks
                labelsize=6,      # ↓ font size of tick labels
                length=0,         # ↓ length of tick marks
                width=0.4,         # ↓ thickness of tick marks
                pad=0.8
            )
            frame.minorticks_off()
            frame.set_xticks([])
            frame.set_xticklabels([])

        else:
            frame.set_yticklabels([])
            frame.set_xticklabels([])
            frame.set_xticks([])
            frame.set_yticks([])
            frame.minorticks_off()



        # optional code block
        data_location_best = "/dataserver/users/formingworlds/borgmann/ARCiS_retrievals/"
        band_best = np.genfromtxt(data_location_best+done[i][8:]+"_350/emis_limits",usecols=0,skip_header=1)
        sigma_min = np.genfromtxt(data_location_best+done[i][8:]+"_350/emis_limits",usecols=3,skip_header=1)
        flux_best = np.genfromtxt(data_location_best+done[i][8:]+"_350/emis_limits",usecols=4,skip_header=1)
        sigma_plus = np.genfromtxt(data_location_best+done[i][8:]+"_350/emis_limits",usecols=5,skip_header=1)
        frame.plot(band_best,flux_best,linewidth=0.5,label='retrieval',color='red')
        frame.fill_between(band_best,sigma_min,sigma_plus,color='red',alpha=0.3)
        if custom_count == 2:
            frame.legend()
            
        custom_count += 1
      

# add axes to the plot (I feel like I over-engineered this)
master_ax = rasterplot.add_axes([0.1, 0.1, 0.8, 0.8])
master_ax.patch.set_alpha(0)
master_ax.set_zorder(-10)

master_ax.spines['top'].set_visible(False)
master_ax.spines['right'].set_visible(False)
master_ax.spines['left'].set_linewidth(1.5)
master_ax.spines['bottom'].set_linewidth(1.5)

y_centers = [0.83, 0.50, 0.17] 
master_ax.set_ylim(0, 1)
master_ax.set_yticks(y_centers)
master_ax.set_yticklabels(["5", "1", "0.1"])

x_centers = [0.135, 0.365, 0.635, 0.865]
master_ax.set_xlim(0, 1)
master_ax.set_xticks(x_centers)
master_ax.set_xticklabels(["-4", "-1.3", "1.3", "4"])

master_ax.tick_params(labelsize=12)


rasterplot.supxlabel('Mantle $f$O$_2$, $\Delta$ IW')
rasterplot.supylabel('Mantle C/H mass ratio')
rasterplot.suptitle("AGNI spectra for varying mantle $f$O$_2$ and mantle H/C mass ratio")
rasterplot.show()
#rasterplot.savefig('../output/figures/ARCiS_fm_raster.png')